<a href="https://colab.research.google.com/github/alanapooler827/554Homework7/blob/main/ST554_HW7_Pooler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Alana Pooler
<br>
Purpose: Complete Homework 7

The purpose of this homework is to practice fitting MLR and logistic regression models (including penalized or regularized models). We will use the wine quality data set from the UCI Machine Learning Repository.

Input variables (based on physicochemical tests)

* fixed acidity
* volatile acidity
* citric acid
* residual sugar
* chlorides
* free sulfur dioxide
* total sulfur dioxide
* density
* pH
* sulphates
* alcohol


Output variable (based on sensory data): quality (score between 0 and 10)

Rather than try to predict quality, let's make our target variable for fitting multiple linear regression type models alcohol.

For fitting logistic regression type models we'll use the type of wine as the response variable.

First import some packages we will need throughout the notebook.

In [425]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV, LogisticRegression

### Read in wine data sets

Read in the red wine data set and view the first few rows

In [446]:
red = pd.read_csv("winequality-red.csv", sep=";")
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Read in the white wine data set and view the first few rows

In [447]:
white = pd.read_csv("winequality-white.csv", sep=";")
white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


Before combining the two data sets together, we will add a "type" column to each to label the wines. We will make this a binary variable using 0 for red and 1 for white.

In [448]:
red["type"] = 0
white["type"] = 1

Now we will combine the two data sets together and view the first few rows of the new data set.

In [449]:
wine = pd.concat([red, white])
wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


### Split the Data

Now we need to split up the data set into a training and test set. We will use stratified sampling to make sure that there is a similar proportion of white and red wines in the training and test sets.

We will use 80% of the data for the training set and 20% of the data for the test set.

In [510]:
df_train, df_test = train_test_split(
    wine,
    test_size=0.2,
    stratify=wine["type"],
    random_state=42
)

# create X and Y train sets for regression
y_train_reg = df_train["alcohol"]
X_train_reg = df_train.drop(columns=["alcohol"])

# create X and Y test sets for regression
y_test_reg = df_test["alcohol"]
X_test_reg = df_test.drop(columns=["alcohol"])

Before we start building any models, let's look at correlations between variables so that we can get a better idea of the data and which fields will be useful predictors.

We will look at the correlation between our response variable, alcohol, and the other numeric variables.

The variables density and quality have a decently strong correlation with alcohol. The variables total sulfar dioxide, residual sugar, chlorides, and free sulfur dioxides have stronger correlations with alcohol than the rest of the variables, but the correlations are pretty weak.

In [511]:
wine.corr(numeric_only=True)["alcohol"]

,alcohol
fixed acidity,-0.095452
volatile acidity,-0.037640
citric acid,-0.010493
residual sugar,-0.359415
chlorides,-0.256916
free sulfur dioxide,-0.179838
total sulfur dioxide,-0.265740
density,-0.686745
pH,0.121248
sulphates,-0.003029


Let's also create a basic MLR model without using cross validation so that we can retrieve variable importance. First, we need to scale the predictors so that the magnitude of the coefficients accurately reflect their importance.

This shows that density is the most important predictor by far. Some of the variables that have higher correlations with alcohol have lower importance, like total sulfur dioxide.

In [512]:
means = X_train_reg.mean()
sds = X_train_reg.std()

X_train_scaled = (X_train_reg - means) / sds
X_test_scaled = (X_test_reg - means) / sds

model = LinearRegression()
model.fit(X_train_scaled, y_train_reg)

importance = pd.DataFrame({"feature": X_train_reg.columns, "importance": abs(model.coef_)})
importance.sort_values(by="importance", ascending=False)

,feature,importance
7,density,1.954502
3,residual sugar,1.068153
0,fixed acidity,0.662886
11,type,0.473329
8,pH,0.413415
9,sulphates,0.151586
1,volatile acidity,0.136609
10,quality,0.088712
2,citric acid,0.076993
5,free sulfur dioxide,0.055511


## Regression Task

We will train several regression models - multiple linear regression, LASSO, ridge regression, and elastic net. For every model, we will use "alcohol" as the response variable.

First, we will fit four different multiple linear regression models and use CV to select the best one.

The first MLR model will be a full model using all variables in the data set (except for the response variable) as predictors.

In [513]:
# Full model
mlr_full_model = cross_validate(
    LinearRegression(),
    X_train_reg,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error",
    return_estimator=True)

The second will be a reduced model, using only a subset of the columns in the wine data set. I have chosen this subset columns using the correlations and variable importance we looked at earlier.

In [514]:
# create train and test sets containing selected subset of variables
subset = [
    "density",
    "pH",
    "type",
    "sulphates",
    "volatile acidity",
    "fixed acidity",
    "residual sugar"
]

X_train_subset = X_train_reg[subset]
X_test_subset = X_test_reg[subset]

mlr2 = cross_validate(
    LinearRegression(),
    X_train_subset,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error")

The third model will incorporate interaction terms. We will use the same subset of predictors we used in the second model.

Since sugar contributes to the density of the wine, these variables may be dependent on each other, so we will create an interaction term between residual sugar and density.

I also read on google that volatile acidity can have an impact on density, so I included an interaction term for those variables as well.

Whether or not I used scaled predictors had no impact on the model metrics, so I went ahead without scaling the predictors.

In [423]:
# create copies of training and test data
X_train_mlr3 = X_train_subset.copy()
X_test_mlr3 = X_test_subset.copy()

# add new interaction term to training and test data
X_train_mlr3["sugar_x_density"] = (
    X_train_mlr3["residual sugar"] * X_train_mlr3["density"]
)

X_test_mlr3["sugar_x_density"] = (
    X_test_mlr3["residual sugar"] * X_test_mlr3["density"]
)

X_train_mlr3["va_x_density"] = (
    X_train_mlr3["volatile acidity"] * X_train_mlr3["density"]
)

X_test_mlr3["va_x_density"] = (
    X_test_mlr3["volatile acidity"] * X_test_mlr3["density"]
)

# fit model 3
mlr3 = cross_validate(
    LinearRegression(),
    X_train_mlr3,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error")

The fourth and final MLR model will use polynomial terms. We will use the same subset of predictors we used in the second model. We will square the density and residual sugar variables to create polynomial terms, since these variables had the highest correlations with alcohol.

In [419]:
# create copy of training and test sets
X_train_mlr4 = X_train_subset.copy()
X_test_mlr4 = X_test_subset.copy()

# create squared density variable in the training and test sets
X_train_mlr4["density_sq"] = (
    X_train_mlr4["density"] ** 2
)

X_test_mlr4["density_sq"] = (
    X_test_mlr4["density"] ** 2
)

# create squared residual sugar variable in the training and test sets
X_train_mlr4["residual_sugar_sq"] = (
    X_train_mlr4["residual sugar"] ** 2
)

X_test_mlr4["residual_sugar_sq"] = (
    X_test_mlr4["residual sugar"] ** 2
)

# fit model 4
mlr4 = cross_validate(
    LinearRegression(),
    X_train_mlr4,
    y_train_reg,
    cv = 5,
    scoring = "neg_mean_squared_error")

Now we will look at the scores from each of the four MLR models to identify the best one.

The model with the polynomial terms has the lowest root mean squared error out of the four models.

In [515]:
print("Full model RMSE:", round(np.sqrt(-np.mean(mlr_full_model["test_score"])),4))
print("Reduced model RMSE:", round(np.sqrt(-np.mean(mlr2["test_score"])),4))
print("Interaction model RMSE:", round(np.sqrt(-np.mean(mlr3["test_score"])),4))
print("Polynomial model RMSE:", round(np.sqrt(-np.mean(mlr4["test_score"])),4))

Full model RMSE: 0.5165
Reduced model RMSE: 0.5255
Interaction model RMSE: 0.509
Polynomial model RMSE: 0.4476


Let's fit the final polynomial model and save RMSE and MAE for use in the final comparison later.

In [371]:
model = LinearRegression()
model.fit(X_train_mlr4, y_train_reg)

# predict on test set
mlr4_preds = model.predict(X_test_mlr4)

# metrics
mlr4_rmse = np.sqrt(mean_squared_error(y_test_reg, mlr4_preds))
mlr4_mae = mean_absolute_error(y_test_reg, mlr4_preds)

print("RMSE:", mlr4_rmse)
print("MAE:", mlr4_mae)

RMSE: 0.45645609906075846
MAE: 0.3314089857250279


Now we will fit a LASSO model using CV to find the best value of alpha. We already created scaled predictors to find variable importance for the MLR model, so we can use those for the next three models.

Since LASSO models already shrink less important variables to zero, we will use all predictors in the model rather than choosing a subset.

In [393]:
# fit LASSO model with CV
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train_reg)

# calculate metrics and get predictions using test set
lasso_preds = lasso.predict(X_test_scaled)
lasso_rmse = np.sqrt(mean_squared_error(y_test_reg, lasso_preds))
lasso_mae = mean_absolute_error(y_test_reg, lasso_preds)

print("Best alpha:", lasso.alpha_)
print("LASSO Test RMSE:", lasso_rmse)
print("LASSO Test MAE:", lasso_mae)

Best alpha: 0.0008254926820107834
LASSO Test RMSE: 0.47708894738144514
LASSO Test MAE: 0.35271503208626676


Now we will fit a Ridge Regression model using cross validation to find the optimal value of alpha for the model. For this model, using all predictors gave slightly better results than using a subset of predictors, so we will leave all of the predictors in this model.

In [394]:
# fit Ridge with CV
ridge = RidgeCV(alphas=np.logspace(-6, 6, 13), cv=5)
# use scaled predictors to fit the model
ridge.fit(X_train_scaled, y_train_reg)

# best alpha
best_alpha = ridge.alpha_

# get predictions for test set
ridge_preds = ridge.predict(X_test_scaled)

# calculate metrics
ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_preds))
ridge_mae = mean_absolute_error(y_test_reg, ridge_preds)

print("Best alpha:", best_alpha)
print("RMSE:", ridge_rmse)
print("MAE:", ridge_mae)

Best alpha: 10.0
RMSE: 0.4672945092451905
MAE: 0.35303505447593775


Lastly, we will fit an elastic net model, once again using cross validation to find the best alpha and L1 ratio parameters for the model. Like the ridge regression model, we will use all predictors for this model since using all predictors performed better than using a subset.

In [395]:
elastic = ElasticNetCV(cv=5, random_state=42)
# use scaled predictors to fit the model
elastic.fit(X_train_scaled, y_train_reg)

# get best parameters
best_alpha = elastic.alpha_
best_l1_ratio = elastic.l1_ratio_

# get predictions for test set
en_preds = elastic.predict(X_test_scaled)

# calculate metrics
en_rmse = np.sqrt(mean_squared_error(y_test_reg, en_preds))
en_mae = mean_absolute_error(y_test_reg, en_preds)

print("Best alpha:", best_alpha)
print("Best l1_ratio:", best_l1_ratio)
print("RMSE:", en_rmse)
print("MAE:", en_mae)

Best alpha: 0.0016509853640215668
Best l1_ratio: 0.5
RMSE: 0.46673632576568397
MAE: 0.3522053056275102


### Regression Model Comparison

Now that we have built all of our models, we can compare them to one another using RMSE and MAE.

The MLR model with polynomial terms has the lowest RMSE and MAE out of the four models, but only by a very small amount.

In [396]:
print("MLR RMSE:", mlr4_rmse)
print("LASSO RMSE:", lasso_rmse)
print("Ridge RMSE:", ridge_rmse)
print("Elastic Net RMSE:", en_rmse)

MLR RMSE: 0.45645609906075846
LASSO RMSE: 0.47708894738144514
Ridge RMSE: 0.4672945092451905
Elastic Net RMSE: 0.46673632576568397


In [397]:
print("MLR MAE:", mlr4_mae)
print("LASSO MAE:", lasso_mae)
print("Ridge MAE:", ridge_mae)
print("Elastic Net MAE:", en_mae)

MLR MAE: 0.3314089857250279
LASSO MAE: 0.35271503208626676
Ridge MAE: 0.35303505447593775
Elastic Net MAE: 0.3522053056275102


## Classification Task

Now we will use a similar process of training and testing models to fit logistic regression models to predict the variable "type". I encoded type as a binary variable with 0 for red wine and 1 for white wine.

We will use log-loss as the metric for choosing models during the training process. During the testing portion, we will compare models using log-loss and accuracy.

First, we need to get our training and test data sets. Our data is already split into training and test sets, so we just need to assign our X and y variables.

In [452]:
# create X and Y train sets for classification
y_train_clf = df_train["type"]
X_train_clf = df_train.drop(columns=["type"])

# create X and Y test sets for classification
y_test_clf = df_test["type"]
X_test_clf = df_test.drop(columns=["type"])

Now we will create standardized predictors for use in our models.

In [502]:
# get mean and sd of training set
means = X_train_clf.mean()
sds = X_train_clf.std()

# standardize training and testing set
X_train_clf_scaled = (X_train_clf - means) / sds
X_test_clf_scaled = (X_test_clf - means) / sds


First, we will build a full logistic regression model using all predictors. We will use cross validation to find the best model, using negative log loss and accuracy as our evaluation metrics.

In [503]:
lr_full_model = cross_validate(
    LogisticRegression(max_iter=1000),
    X_train_clf_scaled,
    y_train_clf,
    cv = 5,
    scoring = ["neg_log_loss", "accuracy"]
)

Next, we will build a reduced model using a subset of predictors. To select this subset, let"s look at feature importance from a full model.

In [504]:
# fit a full model
model = LogisticRegression()
model.fit(X_train_scaled, y_train_clf)

# retrieve importance
importance = pd.DataFrame({
    "feature": X_train_clf.columns,
    "importance": model.coef_[0]
}).sort_values("importance", key=abs, ascending=False)

importance

,feature,importance
7,density,-3.412966
3,residual sugar,2.919324
6,total sulfur dioxide,2.660141
1,volatile acidity,-1.146423
10,alcohol,-1.091329
4,chlorides,-0.896589
5,free sulfur dioxide,-0.705021
9,sulphates,-0.651707
8,pH,-0.359000
2,citric acid,0.328622


Based on this feature importance and some manual feature selection (adding/removing predictors and running the model to see the effect on the evaluation metrics), we can select features to include in a reduced model.

In [505]:
# define subset of columns
subset = [
    "density",
    "residual sugar",
    "total sulfur dioxide",
    "volatile acidity",
    "alcohol",
    "chlorides",
    "free sulfur dioxide",
    "sulphates",
    "pH",
    "fixed acidity"
]

X_train_log2 = X_train_clf_scaled[subset].copy()
X_test_log2 = X_test_clf_scaled[subset].copy()

# fit logistic regression model
lr2 = cross_validate(
    LogisticRegression(max_iter=1000),
    X_train_log2,
    y_train_clf,
    cv = 5,
    scoring = ["neg_log_loss", "accuracy"])

Next, we will build the same reduced model, but this time we will add interaction terms. The same logic used to select interaction terms for the MLR model still applies here, so we will use the same interaction terms for the logistic regression model.

In [506]:
# subset predictors
X_train_log3 = X_train_clf_scaled[subset].copy()
X_test_log3 = X_test_clf_scaled[subset].copy()

# create density x residual sugar interaction term in train and test sets
X_train_log3["density_x_sugar"] = (
    X_train_log3["density"] * X_train_log3["residual sugar"]
)

X_test_log3["density_x_sugar"] = (
    X_test_log3["density"] * X_test_log3["residual sugar"]
)

# create density x volatile acidity interaction term in train and test sets
X_train_log3["density_x_va"] = (
    X_train_log3["density"] * X_train_log3["volatile acidity"]
)

X_test_log3["density_x_va"] = (
    X_test_log3["density"] * X_test_log3["volatile acidity"]
)

lr3 = cross_validate(
    LogisticRegression(max_iter=1000),
    X_train_log3,
    y_train_clf,
    cv = 5,
    scoring = ["neg_log_loss", "accuracy"])

0.035241503893132776
0.9936506996372252


Now we will fit a logistic regression model with polynomial terms. We will use the same subset of predictors that we used in the reduced model, but we will add quadratic terms for density and residual sugar.

Using the standardized predictors in this model greatly improved the log loss score, so I went back and updated the other models to also use the standardized predictors.

In [507]:
# subset predictors
X_train_log4 = X_train_clf_scaled[subset].copy()
X_test_log4 = X_test_clf_scaled[subset].copy()

# create polynomial term for density
X_train_log4["density_sq"] = X_train_log4["density"] ** 2
# create polynomial term for sugar
X_train_log4["sugar_sq"] = X_train_log4["residual sugar"] ** 2

# do the same for test set
X_test_log4["density_sq"] = X_test_log4["density"] ** 2
X_test_log4["sugar_sq"] = X_test_log4["residual sugar"] ** 2

lr4 = cross_validate(
    LogisticRegression(max_iter=1000),
    X_train_log4,
    y_train_clf,
    cv = 5,
    scoring = ["neg_log_loss", "accuracy"],
    return_estimator=True)

Now, we will evaluate all four models using log loss and accuracy.

This time, the model with interaction terms performed the best based on log loss and accracy, but like the MLR models, the metrics are all pretty close between the four models.

In [508]:
print("Full model log-loss:", round(-np.mean(lr_full_model["test_neg_log_loss"]),4))
print("Reduced model log-loss:", round(-np.mean(lr2["test_neg_log_loss"]),4))
print("Interaction model log-loss:", round(-np.mean(lr3["test_neg_log_loss"]),4))
print("Polynomial model log-loss:", round(-np.mean(lr4["test_neg_log_loss"]),4))

Full model log-loss: 0.0443
Reduced model log-loss: 0.0451
Interaction model log-loss: 0.0352
Polynomial model log-loss: 0.0356


In [516]:
print("Full model accuracy:", round(np.mean(lr_full_model["test_accuracy"]),4))
print("Reduced model accuracy:", round(np.mean(lr2["test_accuracy"]),4))
print("Interaction model accuracy:", round(np.mean(lr3["test_accuracy"]),4))
print("Polynomial model accuracy:", round(np.mean(lr4["test_accuracy"]),4))

Full model accuracy: 0.9933
Reduced model accuracy: 0.9935
Interaction model accuracy: 0.9937
Polynomial model accuracy: 0.9937
